In [4]:
import pandas as pd

df = pd.read_csv('../data/raw/superstore.csv', encoding='latin1')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [ ]:
#Data Cleaning

In [6]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [10]:
#Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')

In [11]:
#Convert data types
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

In [12]:
#Handle null values
df.isnull().sum()

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub-category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

In [13]:
df = df.dropna()

In [20]:
#Compute new columns
df['profit_margin'] = df['profit']/df['sales']
df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days
df['year'] = df['order_date'].dt.year
yearly_sales = df.groupby('year')['sales'].sum()
df['yoy_growth'] = df['year'].map(yearly_sales.pct_change())

In [23]:
#Save the cleaned dataset
df.to_csv('../data/clean/clean_superstore.csv', index = False)

In [34]:
#Connect to SQLite Database
from sqlalchemy import create_engine
engine = create_engine("sqlite:///../data/clean/superstore.db")
df.to_sql("superstore", con=engine, if_exists="replace", index = False)

9994

In [35]:
# Read SQL queries from file
with open("../sql/queries.sql", "r") as file:
    sql_script = file.read()
# Split queries by semicolon
queries = [q.strip() for q in sql_script.split(";") if q.strip()]

In [36]:
# Name each query result
query_names = [
    "sales_by_region",
    "profit_by_segment",
    "category_performance",
    "shipping_efficiency"
]

# Run queries and store results
results = {}
for name, query in zip(query_names, queries):
    results[name] = pd.read_sql(query, engine)


In [37]:
# Display results
for name, df_result in results.items():
    print(f"\n{name.upper()}")
    display(df_result)


SALES_BY_REGION


,region,total_sales
0,West,725457.8245
1,East,678781.2400
2,Central,501239.8908
3,South,391721.9050



PROFIT_BY_SEGMENT


,segment,total_profit
0,Consumer,134119.2092
1,Corporate,91979.1340
2,Home Office,60298.6785



CATEGORY_PERFORMANCE


,category,total_sales,total_profit,avg_margin
0,Furniture,741999.7953,18451.2728,0.038784
1,Office Supplies,719047.0320,122490.8008,0.138030
2,Technology,836154.0330,145454.9481,0.156138



SHIPPING_EFFICIENCY


,region,avg_shipping_time
0,Central,4.058115
1,East,3.908708
2,South,3.958025
3,West,3.929753
